# 🏍️ Entrenamiento - Clasificador de cascos

**ACELERADOR RECOMENDADO:** GPU T4 x2

**DEBE ESTAR SUBIDO Y ASOCIADO EL DATASET CORRESPONDIENTE:**
Este script buscará automáticamente el dataset en `/kaggle/input/`

In [ ]:
MODEL_NAME = 'clasificador_v4'

## 1. Instalación de Ultralytics

In [ ]:
!pip install -q ultralytics

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Detectar ruta del Dataset
Vamos a buscar en qué subcarpeta exacta de `/kaggle/input/` quedó montado el dataset.

In [ ]:
import os
from pathlib import Path

input_dir = Path('/kaggle/input')
dataset_root = None
for p in input_dir.rglob('*'):
    if p.is_dir() and p.name in ['train', 'con_casco']:
        dataset_root = p.parent
        break
if dataset_root:
    print(f"✅ Dataset encontrado en: {dataset_root}")
else:
    raise FileNotFoundError("No se encontró la estructura de carpetas correcta en /kaggle/input/")

## 3. Entrenamiento

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-cls.pt')

results = model.train(
    data=str(dataset_root),
    epochs=80,
    patience=20,
    imgsz=64,             # Tamaño del crop de la cabeza en la app
    batch=128,            # Batch enorme porque 64x64 no consume casi nada de VRAM
    device=[0, 1],        # Usa las 2 GPUs (T4 x2) simultáneamente
    amp=True,             # Aceleración por hardware FP16
    
    # OPTIMIZADOR
    optimizer='AdamW',
    lr0=0.0005,           # Learning rate conservador para fine-tuning
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=2.0,
    
    # AUGMENTATIONS
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=10.0,         # Inclinación leve de cabeza
    translate=0.1,
    scale=0.2,            # Variación leve de zoom
    fliplr=0.5,           # Volteado horizontal
    flipud=0.0,
    erasing=0.2,          # Random erasing para evitar memorizar parches/logos
    crop_fraction=1.0,    # Usar todo el crop sin recortar los bordes
    
    # REGULARIZACIÓN ANTI-OVERFITTING
    dropout=0.3,          # 30% dropout en capas finales
    label_smoothing=0.1,  # Suavizado para evitar exceso de confianza
    
    project='/kaggle/working/runs',
    name=MODEL_NAME
)

## 4. Evaluación final en el set de Test
Evaluamos el mejor modelo obtenido sobre el conjunto de test aislado.

In [ ]:
from ultralytics import YOLO
import numpy as np
import glob
import os
from IPython.display import Image, display

print("Evaluando en el set de TEST ambos modelos (best.pt y last.pt)...")
for weight_name in ['best.pt', 'last.pt']:
    weight_path = f'/kaggle/working/runs/{MODEL_NAME}/weights/{weight_name}'
    if not os.path.exists(weight_path):
        print(f"\n⚠️ No se encontró {weight_name}, omitiendo...")
        continue

    print(f"\n==========================================")
    print(f" Cargando modelo para evaluación: {weight_name}")
    print(f"==========================================")
    model_cls = YOLO(weight_path)
    eval_name = f'test_eval_{weight_name.replace(".pt", "")}'
    
    test_results_cls = model_cls.val(data=str(dataset_root), split='test', project=f'/kaggle/working/runs/{MODEL_NAME}', name=eval_name)
    
    try:
        top1 = test_results_cls.results_dict.get('metrics/accuracy_top1', 0)
        top5 = test_results_cls.results_dict.get('metrics/accuracy_top5', 0)
        print(f"Top-1 Accuracy: {top1:.4f}")
        print(f"Top-5 Accuracy: {top5:.4f}")
        
        # --- GUARDAR EN ARCHIVO PARA EL ZIP ---
        eval_dir = f'/kaggle/working/runs/{MODEL_NAME}/{eval_name}'
        os.makedirs(eval_dir, exist_ok=True)
        report_path = os.path.join(eval_dir, f'metricas_detalladas_clasificador_{weight_name.replace(".pt", "")}.txt')
        
        with open(report_path, 'w', encoding='utf-8') as f_out:
            f_out.write(f'==========================================\n')
            f_out.write(f'   MÉTRICAS FINALES - CLASIFICADOR ({weight_name})\n')
            f_out.write(f'==========================================\n')
            f_out.write(f'Top-1 Accuracy: {top1:.4f}\n')
            f_out.write(f'Top-5 Accuracy: {top5:.4f}\n\n')
            
            if hasattr(test_results_cls, 'confusion_matrix'):
                cm = test_results_cls.confusion_matrix.matrix
                classes = model_cls.names
                
                print("\n--- Métricas Detalladas por Clase ---")
                header = f"{'Clase':<20} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}"
                print(header)
                print("-" * 60)
                f_out.write("--- Métricas Detalladas por Clase ---\n")
                f_out.write(header + '\n')
                f_out.write("-" * 60 + '\n')
                
                for i in range(len(classes)):
                    tp = cm[i, i]
                    fp = cm[:, i].sum() - tp
                    fn = cm[i, :].sum() - tp
                    
                    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
                    
                    class_name = classes.get(i, f'Clase {i}')
                    line = f"{class_name:<20} | {precision:<10.4f} | {recall:<10.4f} | {f1:<10.4f}"
                    print(line)
                    f_out.write(line + '\n')
            
            f_out.write('\nDiccionario completo de resultados (bruto):\n')
            for k, v in test_results_cls.results_dict.items():
                f_out.write(f'{k}: {v}\n')
                
        print(f"\n✅ Métricas detalladas guardadas en texto en: {report_path}")
    except Exception as e:
        print(f"Error guardando metricas: {e}")

    # Mostrar gráficos generados por YOLO en el Notebook
    image_files = glob.glob(f'{eval_dir}/*.png')
    for img_path in image_files:
        if any(name in img_path for name in ['confusion_matrix', 'results', 'normalized']):
            display(Image(filename=img_path, width=700))


## 5. Exportar y Descargar el Mejor Modelo

In [ ]:
import shutil

folder_to_zip = f'/kaggle/working/runs/{MODEL_NAME}'
zip_name = f'/kaggle/working/{MODEL_NAME}'

shutil.make_archive(zip_name, 'zip', folder_to_zip)
print("✅ Entrenamiento terminado. Ve a la pestaña 'Output' a la derecha y descarga el archivo f'{MODEL_NAME}.zip'.")